# Notebook 6 — Encoder–Decoder LSTM para predicción meteorológica multistep

**Curso:** Deep Learning  
**Actividad:** 21 — Actividades experimentales  
**Estudiantes:** Juan David Tejedor Medina y Miguel Guerardo Moreno Aveldaño  
**Fecha:** agosto de 2026

**Aplicación:** predecir varias horas futuras de temperatura a partir de una secuencia meteorológica multivariada.

La arquitectura principal es:

\[
\text{Entrada multivariada}
\rightarrow
\text{Encoder recurrente}
\rightarrow
\text{Representación temporal}
\rightarrow
\text{Decoder recurrente}
\rightarrow
\text{temperaturas futuras}
\]

El cuaderno compara de forma controlada el tamaño de entrada y salida, la capacidad de la red, las variables de entrada, LSTM frente a GRU y una versión con Attention.


### Edición para portafolio

Trabajo académico recuperado de la especialización. Se conservan el código, las explicaciones y las atribuciones originales. Se retiraron las salidas y los metadatos de ejecución para facilitar su lectura y revisión. Las conclusiones conservadas pertenecen a la entrega original; los entrenamientos de Deep Learning no se repitieron al organizar este repositorio. Ver [procedencia y autoría](../../docs/PROCEDENCIA.md).


## Objetivos

- Construir ventanas sequence-to-sequence.
- Comprender encoder, contexto y decoder.
- Implementar `RepeatVector` y `TimeDistributed`.
- Comparar con líneas base.
- Evaluar MAE, RMSE, \(R^2\) y error por horizonte.
- Identificar el cuello de botella del vector de contexto.


## 1. Entorno

In [ ]:
import os, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Dispositivos:", tf.config.list_physical_devices())


## 2. Parámetros del problema

- Entrada: 24 horas.
- Salida: 24 horas.
- Variables: temperatura, presión, humedad, viento, precipitación, tiempo cíclico y estación.


In [ ]:
N_DAYS = 365
INPUT_LENGTH = 24
OUTPUT_LENGTH = 24
START_DATE = "2024-01-01"

stations = {
    "Montaña": {"temp_offset": -4.0, "pressure": 780.0, "amp": 6.0, "humidity": 8.0},
    "Urbana": {"temp_offset": 2.0, "pressure": 850.0, "amp": 5.0, "humidity": -4.0},
    "Valle": {"temp_offset": 0.0, "pressure": 820.0, "amp": 7.0, "humidity": 3.0},
}


## 3. Generación de datos meteorológicos sintéticos

In [ ]:
def generate_weather(name, cfg, seed):
    rng = np.random.default_rng(seed)
    n = N_DAYS * 24
    ts = pd.date_range(START_DATE, periods=n, freq="h")
    hour = ts.hour.to_numpy()
    day = ts.dayofyear.to_numpy()
    idx = np.arange(n)

    daily = np.sin(2*np.pi*(hour-6)/24)
    annual = np.sin(2*np.pi*(day-80)/365.25)
    synoptic = np.sin(2*np.pi*idx/(24*7))

    pressure = cfg["pressure"] + 5*synoptic + 2*np.cos(2*np.pi*hour/24) + rng.normal(0,1.3,n)
    humidity = 68 - 18*daily - 8*annual + cfg["humidity"] + rng.normal(0,4,n)
    humidity = np.clip(humidity,20,100)

    wind = 2.5 + 1.2*np.maximum(daily,0) + 0.8*np.abs(synoptic) + rng.gamma(1.5,0.5,n)
    rain_prob = 1/(1+np.exp(-(0.10*(humidity-75)-0.25*(pressure-cfg["pressure"]))))
    rain = rng.binomial(1, np.clip(rain_prob,0,1))*rng.gamma(1.2,1.4,n)

    temperature = (
        16 + cfg["temp_offset"] + cfg["amp"]*daily + 4*annual
        - 0.08*(humidity-60) + 0.10*(pressure-cfg["pressure"])
        - 0.25*rain + rng.normal(0,0.8,n)
    )

    return pd.DataFrame({
        "timestamp": ts, "station": name, "temperature": temperature,
        "pressure": pressure, "humidity": humidity,
        "wind_speed": wind, "precipitation": rain
    })

weather_df = pd.concat(
    [generate_weather(name, cfg, SEED+i) for i,(name,cfg) in enumerate(stations.items())],
    ignore_index=True
)

weather_df.head()


In [ ]:
print(weather_df.shape)
weather_df.groupby("station")[["temperature","pressure","humidity"]].mean().round(2)


## 4. Visualización

In [ ]:
plt.figure(figsize=(14,5))
for station in stations:
    sample = weather_df[weather_df.station == station].head(14*24)
    plt.plot(sample.timestamp, sample.temperature, label=station)
plt.ylabel("Temperatura")
plt.title("Dos semanas de datos horarios")
plt.legend()
plt.grid(alpha=.25)
plt.show()


## 5. Variables temporales y codificación de estación

In [ ]:
weather_df = weather_df.sort_values(["station","timestamp"]).reset_index(drop=True)
weather_df["hour"] = weather_df.timestamp.dt.hour
weather_df["day_of_year"] = weather_df.timestamp.dt.dayofyear

weather_df["hour_sin"] = np.sin(2*np.pi*weather_df.hour/24)
weather_df["hour_cos"] = np.cos(2*np.pi*weather_df.hour/24)
weather_df["day_sin"] = np.sin(2*np.pi*weather_df.day_of_year/365.25)
weather_df["day_cos"] = np.cos(2*np.pi*weather_df.day_of_year/365.25)

dummies = pd.get_dummies(weather_df.station, prefix="station", dtype=float)
model_df = pd.concat([weather_df, dummies], axis=1)

station_cols = dummies.columns.tolist()
feature_cols = [
    "temperature","pressure","humidity","wind_speed","precipitation",
    "hour_sin","hour_cos","day_sin","day_cos"
] + station_cols

print(feature_cols)


## 6. División cronológica

No se mezclan observaciones futuras con pasadas.

- 70 % entrenamiento.
- 15 % validación.
- 15 % prueba.


In [ ]:
train_parts, val_parts, test_parts = [], [], []

for station in stations:
    df = model_df[model_df.station == station].sort_values("timestamp").reset_index(drop=True)
    n = len(df)
    a, b = int(.70*n), int(.85*n)
    train_parts.append(df.iloc[:a])
    val_parts.append(df.iloc[a:b])
    test_parts.append(df.iloc[b:])

train_df = pd.concat(train_parts, ignore_index=True)
val_df = pd.concat(val_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

print(train_df.shape, val_df.shape, test_df.shape)


## 7. Escalamiento

In [ ]:
feature_scaler = StandardScaler().fit(train_df[feature_cols])
target_scaler = StandardScaler().fit(train_df[["temperature"]])


## 8. Ventanas multistep

Cada entrada tiene forma:

\[
(24,F)
\]

y cada salida:

\[
(24,1)
\]


In [ ]:
def create_sequences(df):
    X, y, metadata = [], [], []

    for station, part in df.groupby("station"):
        part = part.sort_values("timestamp").reset_index(drop=True)
        xs = feature_scaler.transform(part[feature_cols])
        ys = target_scaler.transform(part[["temperature"]]).ravel()

        max_start = len(part) - INPUT_LENGTH - OUTPUT_LENGTH + 1

        for start in range(max_start):
            input_end = start + INPUT_LENGTH
            output_end = input_end + OUTPUT_LENGTH

            X.append(xs[start:input_end])
            y.append(ys[input_end:output_end].reshape(OUTPUT_LENGTH,1))
            metadata.append({
                "station": station,
                "forecast_start": part.loc[input_end,"timestamp"],
                "last_temperature": part.loc[input_end-1,"temperature"],
                "future_temperature": part.loc[input_end:output_end-1,"temperature"].to_numpy()
            })

    return np.asarray(X,np.float32), np.asarray(y,np.float32), pd.DataFrame(metadata)

X_train, y_train, meta_train = create_sequences(train_df)
X_val, y_val, meta_val = create_sequences(val_df)
X_test, y_test, meta_test = create_sequences(test_df)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)


## 9. Ejemplo de entrada y salida

In [ ]:
i = 0
temp_idx = feature_cols.index("temperature")
historical = X_train[i,:,temp_idx]*feature_scaler.scale_[temp_idx] + feature_scaler.mean_[temp_idx]
future = meta_train.loc[i,"future_temperature"]

plt.figure(figsize=(11,5))
plt.plot(range(-24,0), historical, marker="o", label="Entrada")
plt.plot(range(1,25), future, marker="o", label="Objetivo")
plt.axvline(0, linestyle="--")
plt.xlabel("Hora relativa")
plt.ylabel("Temperatura")
plt.legend()
plt.grid(alpha=.25)
plt.show()


## 10. Líneas base

**Persistencia:** todas las horas futuras se igualan a la última temperatura observada.

**Patrón diario:** se reutilizan las 24 temperaturas de la ventana anterior.


In [ ]:
def inverse_target(values):
    shape = values.shape
    return target_scaler.inverse_transform(values.reshape(-1,1)).reshape(shape)

y_test_real = inverse_target(y_test).squeeze(-1)

persistence_pred = np.repeat(
    meta_test.last_temperature.to_numpy().reshape(-1,1),
    OUTPUT_LENGTH,
    axis=1
)

temp_idx = feature_cols.index("temperature")
daily_pred = X_test[:,:,temp_idx]*feature_scaler.scale_[temp_idx] + feature_scaler.mean_[temp_idx]


In [ ]:
def metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true.ravel(), y_pred.ravel())
    mse = mean_squared_error(y_true.ravel(), y_pred.ravel())
    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true.ravel(), y_pred.ravel())
    }

baseline_df = pd.DataFrame([
    {"Modelo":"Persistencia", **metrics(y_test_real,persistence_pred)},
    {"Modelo":"Patrón diario", **metrics(y_test_real,daily_pred)}
])
baseline_df


# Parte V — Encoder–Decoder LSTM

El encoder procesa la secuencia y produce un vector de contexto.

`RepeatVector(24)` replica el contexto.

El decoder genera una secuencia de 24 estados.

`TimeDistributed(Dense(1))` produce una temperatura por hora.


In [ ]:
LATENT_UNITS = 64

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=X_train.shape[1:], name="encoder_input"),
    tf.keras.layers.LSTM(LATENT_UNITS, name="encoder"),
    tf.keras.layers.RepeatVector(OUTPUT_LENGTH, name="repeat_context"),
    tf.keras.layers.LSTM(LATENT_UNITS, return_sequences=True, name="decoder"),
    tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(32, activation="relu"),
        name="hidden_output"
    ),
    tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(1),
        name="temperature_output"
    )
], name="encoder_decoder_weather")

model.summary()


## 11. Compilación

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="mse",
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(name="mae"),
        tf.keras.metrics.RootMeanSquaredError(name="rmse")
    ]
)


## 12. Callbacks y entrenamiento

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=7, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=.5, patience=3, min_lr=1e-6
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "best_encoder_decoder_weather.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

start = time.perf_counter()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val,y_val),
    epochs=60,
    batch_size=128,
    callbacks=callbacks,
    verbose=0
)

training_time = time.perf_counter() - start
print("Tiempo:", training_time)

print(f"Épocas ejecutadas: {len(history.history['loss'])}")


## 13. Curvas de aprendizaje

In [ ]:
hist = pd.DataFrame(history.history)

plt.figure(figsize=(8,5))
plt.plot(hist.loss, label="Entrenamiento")
plt.plot(hist.val_loss, label="Validación")
plt.xlabel("Época")
plt.ylabel("MSE escalado")
plt.legend()
plt.grid(alpha=.25)
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(hist.mae, label="MAE entrenamiento")
plt.plot(hist.val_mae, label="MAE validación")
plt.xlabel("Época")
plt.ylabel("MAE escalado")
plt.legend()
plt.grid(alpha=.25)
plt.show()


## 14. Evaluación global

In [ ]:
pred_scaled = model.predict(X_test, batch_size=128, verbose=0)
pred_real = inverse_target(pred_scaled).squeeze(-1)

model_metrics = metrics(y_test_real, pred_real)

results = pd.concat([
    baseline_df,
    pd.DataFrame([{"Modelo":"Encoder–Decoder LSTM", **model_metrics}])
], ignore_index=True)

results


In [ ]:
plt.figure(figsize=(9,5))
plt.bar(results.Modelo, results.RMSE)
plt.ylabel("RMSE")
plt.title("Comparación de modelos")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=.25)
plt.show()


## 15. Error por horizonte

In [ ]:
rows = []

for h in range(OUTPUT_LENGTH):
    true_h = y_test_real[:,h]
    pred_h = pred_real[:,h]
    rows.append({
        "Hora": h+1,
        "MAE": mean_absolute_error(true_h,pred_h),
        "RMSE": np.sqrt(mean_squared_error(true_h,pred_h)),
        "R2": r2_score(true_h,pred_h)
    })

horizon_df = pd.DataFrame(rows)
horizon_df.head()


In [ ]:
plt.figure(figsize=(10,5))
plt.plot(horizon_df.Hora, horizon_df.RMSE, marker="o")
plt.xlabel("Horizonte de pronóstico (horas)")
plt.ylabel("RMSE")
plt.title("Error por horizonte")
plt.grid(alpha=.25)
plt.show()


## 16. Visualización de un pronóstico

In [ ]:
i = 0

plt.figure(figsize=(11,5))
plt.plot(range(1,25), y_test_real[i], marker="o", label="Real")
plt.plot(range(1,25), pred_real[i], marker="o", label="Encoder–Decoder")
plt.plot(range(1,25), persistence_pred[i], linestyle="--", label="Persistencia")
plt.plot(range(1,25), daily_pred[i], linestyle=":", label="Patrón diario")
plt.xlabel("Hora futura")
plt.ylabel("Temperatura")
plt.title(f"Estación: {meta_test.loc[i,'station']}")
plt.legend()
plt.grid(alpha=.25)
plt.show()


## 17. Evaluación por estación

In [ ]:
station_rows = []

for station in stations:
    mask = meta_test.station.to_numpy() == station
    station_rows.append({
        "Estación": station,
        **metrics(y_test_real[mask], pred_real[mask])
    })

station_df = pd.DataFrame(station_rows)
station_df


## 18. Residuos

In [ ]:
residuals = y_test_real - pred_real

plt.figure(figsize=(8,5))
plt.hist(residuals.ravel(), bins=40, edgecolor="black")
plt.xlabel("Residuo")
plt.ylabel("Frecuencia")
plt.title("Distribución de residuos")
plt.grid(alpha=.2)
plt.show()


In [ ]:
mean_residual = residuals.mean(axis=0)

plt.figure(figsize=(10,5))
plt.plot(range(1,25), mean_residual, marker="o")
plt.axhline(0, linestyle="--")
plt.xlabel("Horizonte")
plt.ylabel("Residuo medio")
plt.title("Sesgo por horizonte")
plt.grid(alpha=.25)
plt.show()


## 19. Vector de contexto

El encoder comprime toda la secuencia en un vector de 64 componentes.

Este vector fijo puede convertirse en un cuello de botella cuando las secuencias son largas.


In [ ]:
context_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=model.get_layer("encoder").output
)

contexts = context_model.predict(X_test[:100], verbose=0)
print("Forma del contexto:", contexts.shape)


## 20. ¿Por qué aparece Attention?

En el Encoder–Decoder clásico:

\[
\text{toda la entrada}
\rightarrow
\text{un único vector}
\]

Con Attention, cada paso del decoder puede consultar directamente todos los estados del encoder.

Esta será la transición hacia Transformers.


## 21. Actividades experimentales

Se usa como **control** el Encoder–Decoder LSTM original: entrada de 24 horas, salida de 24 horas, 64 unidades latentes y todas las variables. En cada bloque se modifica una sola condición:

1. Entrada de 12, 48 y 72 horas.
2. Salida de 6, 12 y 48 horas.
3. Capacidad de 32, 64 y 128 unidades latentes.
4. Solo temperatura frente a todas las variables.
5. LSTM frente a GRU.
6. Encoder–Decoder clásico frente a Encoder–Decoder con Attention.

Para que la comparación sea reproducible se fijan semillas, se mantienen la división cronológica, el escalamiento, el optimizador, el tamaño de lote y los callbacks. Se toman ventanas cada tres horas para conservar toda la cobertura temporal con un costo computacional razonable. La selección del modelo se hace con validación; la prueba se usa únicamente para la evaluación final.


### 21.1 Funciones para crear ventanas y arquitecturas comparables

La función de ventanas acepta diferentes longitudes de entrada, horizontes de salida y conjuntos de variables. El modelo clásico comprime la entrada en un vector fijo. La versión con Attention conserva todos los estados del encoder y permite que cada paso del decoder los consulte.


In [ ]:
import gc

EXPERIMENT_STRIDE = 3
MAX_EPOCHS = 25
EXPERIMENT_BATCH_SIZE = 128

def create_experiment_sequences(df, input_length, output_length, selected_features,
                                stride=EXPERIMENT_STRIDE):
    # Crea ventanas cronológicas sin mezclar estaciones ni usar datos futuros.
    X, y, metadata = [], [], []
    selected_idx = [feature_cols.index(col) for col in selected_features]

    for station, part in df.groupby("station", sort=False):
        part = part.sort_values("timestamp").reset_index(drop=True)
        all_scaled = feature_scaler.transform(part[feature_cols])
        xs = all_scaled[:, selected_idx]
        ys = target_scaler.transform(part[["temperature"]]).ravel()
        max_start = len(part) - input_length - output_length + 1

        for start in range(0, max_start, stride):
            input_end = start + input_length
            output_end = input_end + output_length
            X.append(xs[start:input_end])
            y.append(ys[input_end:output_end, None])
            metadata.append((station, part.loc[input_end, "timestamp"]))

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        pd.DataFrame(metadata, columns=["station", "forecast_start"]),
    )


def build_seq2seq(input_shape, output_length, latent_units=64,
                  recurrent_type="LSTM", use_attention=False):
    # Construye modelos equivalentes variando solo la condición indicada.
    inputs = tf.keras.Input(shape=input_shape, name="encoder_input")

    if recurrent_type == "GRU":
        encoder = tf.keras.layers.GRU(
            latent_units, return_sequences=True, return_state=True, name="encoder_gru"
        )
        encoder_sequence, encoder_state = encoder(inputs)
        repeated = tf.keras.layers.RepeatVector(output_length, name="repeat_context")(encoder_state)
        decoder_sequence = tf.keras.layers.GRU(
            latent_units, return_sequences=True, name="decoder_gru"
        )(repeated, initial_state=encoder_state)
    else:
        encoder = tf.keras.layers.LSTM(
            latent_units, return_sequences=True, return_state=True, name="encoder_lstm"
        )
        encoder_sequence, state_h, state_c = encoder(inputs)
        repeated = tf.keras.layers.RepeatVector(output_length, name="repeat_context")(state_h)
        decoder_sequence = tf.keras.layers.LSTM(
            latent_units, return_sequences=True, name="decoder_lstm"
        )(repeated, initial_state=[state_h, state_c])

    representation = decoder_sequence
    if use_attention:
        attended = tf.keras.layers.Attention(name="temporal_attention")(
            [decoder_sequence, encoder_sequence]
        )
        representation = tf.keras.layers.Concatenate(name="decoder_plus_context")(
            [decoder_sequence, attended]
        )

    hidden = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(32, activation="relu"), name="hidden_output"
    )(representation)
    outputs = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(1), name="temperature_output"
    )(hidden)

    model_name = "encoder_decoder_attention" if use_attention else f"encoder_decoder_{recurrent_type.lower()}"
    experiment_model = tf.keras.Model(inputs, outputs, name=model_name)
    experiment_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return experiment_model


def architecture_label(config):
    attention_text = " + Attention" if config.get("attention", False) else ""
    return (
        f"Encoder–Decoder {config['recurrent']}{attention_text}; "
        f"{config['latent']} unidades; Dense(32); salida TimeDistributed(1)"
    )


print("Funciones experimentales preparadas.")


### 21.2 Diseño experimental

La lista siguiente contiene las configuraciones únicas. La fila de control se reutiliza en todas las comparaciones; por eso no se entrena varias veces la misma red. Las métricas se calculan en unidades reales de temperatura después de invertir el escalamiento.


In [ ]:
ALL_FEATURES = feature_cols.copy()
TEMP_ONLY = ["temperature"]

experiment_configs = [
    dict(key="control", experiment="Control", configuration="24→24 h, LSTM 64, todas",
         input_length=24, output_length=24, latent=64, features=ALL_FEATURES,
         recurrent="LSTM", attention=False),

    dict(key="input_12", experiment="A: entrada", configuration="12 horas",
         input_length=12, output_length=24, latent=64, features=ALL_FEATURES,
         recurrent="LSTM", attention=False),
    dict(key="input_48", experiment="A: entrada", configuration="48 horas",
         input_length=48, output_length=24, latent=64, features=ALL_FEATURES,
         recurrent="LSTM", attention=False),
    dict(key="input_72", experiment="A: entrada", configuration="72 horas",
         input_length=72, output_length=24, latent=64, features=ALL_FEATURES,
         recurrent="LSTM", attention=False),

    dict(key="output_6", experiment="B: salida", configuration="6 horas",
         input_length=24, output_length=6, latent=64, features=ALL_FEATURES,
         recurrent="LSTM", attention=False),
    dict(key="output_12", experiment="B: salida", configuration="12 horas",
         input_length=24, output_length=12, latent=64, features=ALL_FEATURES,
         recurrent="LSTM", attention=False),
    dict(key="output_48", experiment="B: salida", configuration="48 horas",
         input_length=24, output_length=48, latent=64, features=ALL_FEATURES,
         recurrent="LSTM", attention=False),

    dict(key="latent_32", experiment="C: unidades", configuration="32 unidades",
         input_length=24, output_length=24, latent=32, features=ALL_FEATURES,
         recurrent="LSTM", attention=False),
    dict(key="latent_128", experiment="C: unidades", configuration="128 unidades",
         input_length=24, output_length=24, latent=128, features=ALL_FEATURES,
         recurrent="LSTM", attention=False),

    dict(key="temperature_only", experiment="D: variables", configuration="Solo temperatura",
         input_length=24, output_length=24, latent=64, features=TEMP_ONLY,
         recurrent="LSTM", attention=False),

    dict(key="gru", experiment="E: recurrente", configuration="GRU",
         input_length=24, output_length=24, latent=64, features=ALL_FEATURES,
         recurrent="GRU", attention=False),

    dict(key="attention", experiment="F: Attention", configuration="LSTM + Attention",
         input_length=24, output_length=24, latent=64, features=ALL_FEATURES,
         recurrent="LSTM", attention=True),
]

design_df = pd.DataFrame([
    {
        "Clave": c["key"],
        "Experimento": c["experiment"],
        "Configuración": c["configuration"],
        "Entrada (h)": c["input_length"],
        "Salida (h)": c["output_length"],
        "Variables": len(c["features"]),
        "Unidades": c["latent"],
        "Celda": c["recurrent"],
        "Attention": "Sí" if c["attention"] else "No",
    }
    for c in experiment_configs
])
design_df


### 21.3 Entrenamiento y evaluación

Se utiliza `EarlyStopping` para evitar sobreentrenamiento y restaurar los mejores pesos según la pérdida de validación. `ReduceLROnPlateau` disminuye la tasa de aprendizaje cuando el avance se estanca. La prueba permanece aislada hasta el cálculo de las métricas finales.


In [ ]:
experiment_rows = []
experiment_histories = {}
horizon_profiles = {}
forecast_examples = {}

for number, config in enumerate(experiment_configs, start=1):
    print(f"[{number:02d}/{len(experiment_configs)}] {config['experiment']} — {config['configuration']}")
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)

    X_tr, y_tr, _ = create_experiment_sequences(
        train_df, config["input_length"], config["output_length"], config["features"]
    )
    X_va, y_va, _ = create_experiment_sequences(
        val_df, config["input_length"], config["output_length"], config["features"]
    )
    X_te, y_te, meta_te = create_experiment_sequences(
        test_df, config["input_length"], config["output_length"], config["features"]
    )

    experiment_model = build_seq2seq(
        X_tr.shape[1:],
        config["output_length"],
        latent_units=config["latent"],
        recurrent_type=config["recurrent"],
        use_attention=config["attention"],
    )
    parameter_count = experiment_model.count_params()

    experiment_callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6
        ),
    ]

    start_time = time.perf_counter()
    experiment_history = experiment_model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=MAX_EPOCHS,
        batch_size=EXPERIMENT_BATCH_SIZE,
        callbacks=experiment_callbacks,
        verbose=0,
    )
    elapsed = time.perf_counter() - start_time

    prediction_scaled = experiment_model.predict(
        X_te, batch_size=EXPERIMENT_BATCH_SIZE, verbose=0
    )
    truth_real = inverse_target(y_te).squeeze(-1)
    prediction_real = inverse_target(prediction_scaled).squeeze(-1)
    test_metrics = metrics(truth_real, prediction_real)

    horizon_rmse = [
        np.sqrt(mean_squared_error(truth_real[:, h], prediction_real[:, h]))
        for h in range(config["output_length"])
    ]

    experiment_rows.append({
        "Clave": config["key"],
        "Experimento": config["experiment"],
        "Configuración": config["configuration"],
        "Arquitectura": architecture_label(config),
        "Entrada_h": config["input_length"],
        "Salida_h": config["output_length"],
        "Variables": "Solo temperatura" if len(config["features"]) == 1 else "Todas (12)",
        "Parámetros": parameter_count,
        "Épocas": len(experiment_history.history["loss"]),
        "Tiempo_s": elapsed,
        **test_metrics,
    })
    experiment_histories[config["key"]] = pd.DataFrame(experiment_history.history)
    horizon_profiles[config["key"]] = horizon_rmse
    forecast_examples[config["key"]] = {
        "truth": truth_real[0].copy(),
        "prediction": prediction_real[0].copy(),
        "station": meta_te.iloc[0]["station"],
    }
    print(
        f"    parámetros={parameter_count:,} | épocas={len(experiment_history.history['loss'])} | "
        f"MAE={test_metrics['MAE']:.4f} | RMSE={test_metrics['RMSE']:.4f} | "
        f"R²={test_metrics['R2']:.4f}"
    )

    del experiment_model, X_tr, y_tr, X_va, y_va, X_te, y_te
    gc.collect()

experiment_results = pd.DataFrame(experiment_rows)
print("\nEntrenamientos terminados:", len(experiment_results))


### 21.4 Tabla consolidada de resultados

Un RMSE y un MAE menores indican predicciones más cercanas a los valores reales. Un (R^2) mayor indica que el modelo explica mejor la variabilidad de la temperatura. Los parámetros, las épocas y las métricas permiten comparar capacidad y desempeño; el tiempo total depende del hardware usado.


In [ ]:
result_columns = [
    "Experimento", "Configuración", "Arquitectura", "Entrada_h", "Salida_h",
    "Variables", "Parámetros", "Épocas", "MAE", "RMSE", "R2"
]
formatted_results = experiment_results[result_columns].copy()
formatted_results[["MAE", "RMSE", "R2"]] = formatted_results[["MAE", "RMSE", "R2"]].round(4)
formatted_results


### 21.5 Comparaciones visuales

La primera gráfica resume el RMSE de todas las configuraciones. Después se relaciona la capacidad con el error, se comparan por separado las variables modificadas y se contrastan los modelos recurrentes principales. Las curvas de aprendizaje y el error por horizonte del modelo base permanecen en las secciones 13 y 15.


In [ ]:
plot_df = experiment_results.sort_values("RMSE", ascending=True)

plt.figure(figsize=(11, 7))
colors = ["#2a9d8f" if key == "control" else "#457b9d" for key in plot_df["Clave"]]
plt.barh(plot_df["Experimento"] + " | " + plot_df["Configuración"], plot_df["RMSE"], color=colors)
plt.xlabel("RMSE de prueba")
plt.title("Actividad 21 — comparación global")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 6))
for _, row in experiment_results.iterrows():
    plt.scatter(row["Parámetros"], row["RMSE"], s=65)
    plt.annotate(row["Configuración"], (row["Parámetros"], row["RMSE"]),
                 xytext=(4, 4), textcoords="offset points", fontsize=8)
plt.xscale("log")
plt.xlabel("Número de parámetros (escala logarítmica)")
plt.ylabel("RMSE de prueba")
plt.title("Capacidad del modelo frente a error")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
input_view = pd.concat([experiment_results[experiment_results["Clave"] == "control"], experiment_results[experiment_results["Experimento"] == "A: entrada"]]).sort_values("Entrada_h")
axes[0].plot(input_view["Entrada_h"], input_view["RMSE"], marker="o")
axes[0].set_title("Longitud de entrada"); axes[0].set_xlabel("Horas de entrada")
output_view = pd.concat([experiment_results[experiment_results["Clave"] == "control"], experiment_results[experiment_results["Experimento"] == "B: salida"]]).sort_values("Salida_h")
axes[1].plot(output_view["Salida_h"], output_view["RMSE"], marker="o", color="#e76f51")
axes[1].set_title("Horizonte de salida"); axes[1].set_xlabel("Horas de salida")
unit_view = pd.concat([experiment_results[experiment_results["Clave"] == "control"], experiment_results[experiment_results["Experimento"] == "C: unidades"]]).sort_values("Parámetros")
axes[2].plot([32, 64, 128], unit_view["RMSE"], marker="o", color="#2a9d8f")
axes[2].set_title("Unidades latentes"); axes[2].set_xlabel("Unidades")
for axis in axes:
    axis.set_ylabel("RMSE"); axis.grid(alpha=0.25)
plt.tight_layout(); plt.show()


In [ ]:
key_models = experiment_results[experiment_results["Clave"].isin(["control", "temperature_only", "gru", "attention"])].copy()
labels = ["Control LSTM", "Solo temperatura", "GRU", "Attention"]
x = np.arange(len(key_models)); width = 0.36
plt.figure(figsize=(10, 5))
plt.bar(x - width/2, key_models["MAE"], width, label="MAE")
plt.bar(x + width/2, key_models["RMSE"], width, label="RMSE")
plt.xticks(x, labels, rotation=10); plt.ylabel("Error de prueba")
plt.title("Comparación de modelos clave"); plt.legend(); plt.grid(axis="y", alpha=0.25)
plt.tight_layout(); plt.show()


### 21.6 Análisis automático de los seis experimentos

Las conclusiones siguientes se calculan a partir de las métricas guardadas, por lo que corresponden exactamente a esta ejecución y no a valores escritos manualmente.


In [ ]:
def row_for(key):
    return experiment_results.loc[experiment_results["Clave"] == key].iloc[0]

control_row = row_for("control")
best_global = experiment_results.loc[experiment_results["RMSE"].idxmin()]

input_comparison = pd.concat([
    experiment_results[experiment_results["Clave"] == "control"],
    experiment_results[experiment_results["Experimento"] == "A: entrada"],
]).sort_values("Entrada_h")

output_comparison = pd.concat([
    experiment_results[experiment_results["Clave"] == "control"],
    experiment_results[experiment_results["Experimento"] == "B: salida"],
]).sort_values("Salida_h")

latent_comparison = pd.concat([
    experiment_results[experiment_results["Clave"] == "control"],
    experiment_results[experiment_results["Experimento"] == "C: unidades"],
]).sort_values("Parámetros")

best_input = input_comparison.loc[input_comparison["RMSE"].idxmin()]
best_output = output_comparison.loc[output_comparison["RMSE"].idxmin()]
best_latent = latent_comparison.loc[latent_comparison["RMSE"].idxmin()]
temp_row = row_for("temperature_only")
gru_row = row_for("gru")
attention_row = row_for("attention")

print("CONCLUSIONES DE LA ACTIVIDAD 21\n")
print(
    f"1. Entrada: la mejor longitud fue {int(best_input['Entrada_h'])} horas "
    f"(RMSE={best_input['RMSE']:.4f}, R²={best_input['R2']:.4f}). "
    "Una ventana más larga aporta contexto, pero también puede añadir información redundante y aumentar el costo."
)
print(
    f"2. Salida: el menor RMSE se obtuvo al pronosticar {int(best_output['Salida_h'])} horas "
    f"(RMSE={best_output['RMSE']:.4f}). Los horizontes no son idénticos en dificultad: "
    "a medida que se pronostica más lejos se acumula incertidumbre."
)
print(
    f"3. Unidades: la mejor capacidad probada fue {best_latent['Configuración']} "
    f"con {int(best_latent['Parámetros']):,} parámetros y RMSE={best_latent['RMSE']:.4f}. "
    "Más unidades aumentan la capacidad, pero no garantizan una mejora proporcional."
)
variable_winner = "todas las variables" if control_row["RMSE"] <= temp_row["RMSE"] else "solo temperatura"
print(
    f"4. Variables: ganó {variable_winner}. Control multivariado RMSE={control_row['RMSE']:.4f}; "
    f"solo temperatura RMSE={temp_row['RMSE']:.4f}. Las variables adicionales son útiles solo "
    "si aportan señal predictiva que compense la complejidad adicional."
)
recurrent_winner = "GRU" if gru_row["RMSE"] < control_row["RMSE"] else "LSTM"
print(
    f"5. Celda recurrente: {recurrent_winner} obtuvo el menor RMSE. "
    f"LSTM={control_row['RMSE']:.4f}; GRU={gru_row['RMSE']:.4f}. "
    f"GRU usa {int(gru_row['Parámetros']):,} parámetros frente a {int(control_row['Parámetros']):,} de LSTM."
)
attention_winner = "Attention" if attention_row["RMSE"] < control_row["RMSE"] else "el modelo clásico"
print(
    f"6. Attention: el mejor fue {attention_winner}. Clásico RMSE={control_row['RMSE']:.4f}; "
    f"Attention RMSE={attention_row['RMSE']:.4f}. Attention elimina la dependencia de un único "
    "vector de contexto, aunque su ventaja depende de la longitud, los datos y el ajuste."
)
print(
    f"\nMejor resultado global: {best_global['Experimento']} — {best_global['Configuración']}, "
    f"MAE={best_global['MAE']:.4f}, RMSE={best_global['RMSE']:.4f}, R²={best_global['R2']:.4f}."
)


## 22. Preguntas de reflexión

### 1. ¿Qué diferencia existe entre predicción one-step y multistep?

La predicción **one-step** estima únicamente el valor del siguiente instante. La predicción **multistep** estima varios instantes futuros. Esta última es más exigente porque la incertidumbre suele crecer con el horizonte y debe aprenderse también la relación entre los valores futuros.

### 2. ¿Qué función cumple el encoder?

El encoder recorre la secuencia de entrada y transforma las observaciones meteorológicas en una representación interna. En una LSTM, sus estados oculto y de memoria resumen patrones como ciclos diarios, tendencias y relaciones entre variables.

### 3. ¿Qué contiene el vector de contexto?

Contiene una representación numérica aprendida de la secuencia de entrada. No guarda literalmente las 24 horas, sino características comprimidas que el entrenamiento considera útiles para producir el pronóstico.

### 4. ¿Qué hace `RepeatVector`?

Replica el vector de contexto tantas veces como pasos tenga la salida. Así, un vector de forma `(unidades,)` se convierte en una secuencia `(horizonte, unidades)` que puede recibir el decoder.

### 5. ¿Por qué el decoder usa `return_sequences=True`?

Porque se necesita un estado oculto para **cada** hora futura. Si fuera `False`, el decoder devolvería solo el último estado y no sería posible obtener directamente una secuencia completa de temperaturas.

### 6. ¿Qué hace `TimeDistributed(Dense(1))`?

Aplica la misma capa densa a cada paso temporal de la salida del decoder. Convierte cada estado oculto en un único valor continuo: la temperatura pronosticada para esa hora.

### 7. ¿Por qué se comparan líneas base?

Las líneas base indican el desempeño mínimo que debe superar una red compleja. Si el modelo no mejora estrategias simples como persistencia o patrón diario, su mayor costo no está justificado y puede existir un problema en los datos, el diseño o el entrenamiento.

### 8. ¿Por qué el error suele aumentar con el horizonte?

Los estados más lejanos dependen de fenómenos que todavía no se observan y son menos condicionados por la ventana de entrada. Por eso se acumula incertidumbre y las predicciones tienden a acercarse a patrones promedio.

### 9. ¿Qué significa un residuo positivo?

En este cuaderno el residuo se define como `real − predicción`. Un residuo positivo indica que el valor real fue mayor que el pronosticado; por tanto, el modelo subestimó la temperatura.

### 10. ¿Cuál es la limitación del vector de contexto único?

Obliga a comprimir toda la secuencia en un vector de tamaño fijo. Cuando la entrada es larga o contiene muchos eventos relevantes, parte de la información puede perderse y el decoder no puede volver directamente a un instante concreto.

### 11. ¿Cómo intenta resolverla Attention?

Attention conserva los estados del encoder y calcula, para cada paso del decoder, qué instantes de la entrada son más relevantes. El decoder recibe una combinación ponderada de esos estados en lugar de depender únicamente del último vector.

### 12. ¿Qué ventaja podría aportar un Transformer?

Un Transformer modela dependencias entre posiciones mediante autoatención y procesa muchos pasos en paralelo durante el entrenamiento. Puede capturar relaciones de largo alcance con mayor facilidad, aunque suele requerir más datos, memoria y regularización.


## 23. Síntesis

En esta actividad se construyó un flujo experimental completo para pronóstico meteorológico multistep:

\[
\text{datos cronológicos}
\rightarrow
\text{ventanas entrada/salida}
\rightarrow
\text{Encoder}
\rightarrow
\text{Decoder}
\rightarrow
\text{pronóstico multihorizonte}
\rightarrow
\text{evaluación}
\]

La comparación controlada mostró que la calidad no depende únicamente de hacer la red más grande. También influyen la cantidad de contexto temporal, la distancia del horizonte, la información de las variables, el tipo de celda recurrente y el mecanismo usado para transferir información del encoder al decoder.

El modelo clásico usa un vector de contexto fijo; Attention permite consultar la secuencia codificada completa. La decisión final debe basarse conjuntamente en MAE, RMSE, (R^2), error por horizonte, curvas de validación, número de parámetros y tiempo de entrenamiento.

**Integrantes:** Juan David Tejedor Medina y Miguel Guerardo Moreno Aveldaño.
